In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder
from RMSELoss import RMSELoss
import plotly.graph_objects as go

# Get Dataset

In [3]:
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)


# Augment for New Columns

In [4]:
# Augment the training DataFrame with empty columns for calculations

train_df_c = train_df
train_df_c['FPull_4d_310MPa'] = ''
train_df_c['FPull_5d_310MPa'] = ''
train_df_c['FPull_4d_365MPa'] = ''
train_df_c['FPull_5d_365MPa'] = ''
train_df_c['FPull_4d_440MPa'] = ''
train_df_c['FPull_5d_440MPa'] = ''
train_df_c['FPull_4d_sig_06_365MPa'] = ''
train_df_c['FPull_4d_sig_07_365MPa'] = ''
train_df_c['FPull_4d06_365MPa_nosquare'] = ''
train_df_c['FPull_4d07_365MPa_nosquare'] = ''

# Calculate values

Formula: 

Shear-failure model: 

F = A * τ 

A = pi * d² / 4

τ ≈ 0.8 * σ

Nugget Diameter Relation:

d ≈ k * √t           (with k = 4 or 5)

# F = (pi / 4) * ((4 or 5) * √t)² * (0.8 * σ)

In [18]:
for i in range(train_df_c.shape[0]):

    # Calculate the thickness based on the minimum of either Thickness A or Thickness B
    if train_df_c['Thickness A (mm)'][i] <= train_df_c['Thickness B (mm)'][i]:
        t = train_df_c['Thickness A (mm)'][i]
    else:
        t = train_df_c['Thickness B (mm)'][i]

    # Calculate the pull force for different diameters and yield strengths
    f_pull_4d_310MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.8 * 310)
    train_df_c.loc[i, 'FPull_4d_310MPa'] = round(f_pull_4d_310MPa, 1)
    f_pull_5d_310MPa = (np.pi/4) * np.square(5 * np.sqrt(t)) * (0.8 * 310)
    train_df_c.loc[i, 'FPull_5d_310MPa'] = round(f_pull_5d_310MPa, 1)

    f_pull_4d_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.7 * 365)
    train_df_c.loc[i, 'FPull_4d_365MPa'] = round(f_pull_4d_365MPa, 1)
    f_pull_5d_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.8 * 365)
    train_df_c.loc[i, 'FPull_5d_365MPa'] = round(f_pull_5d_365MPa, 1)

    f_pull_4d06_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.6 * 365)
    train_df_c.loc[i, 'FPull_4d_sig_06_365MPa'] = round(f_pull_4d06_365MPa, 1)
    f_pull_5d07_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.7 * 365)
    train_df_c.loc[i, 'FPull_4d_sig_07_365MPa'] = round(f_pull_5d07_365MPa, 1)

    f_pull_4d06_365MPa_nosquare = (np.pi/4) * np.square(4 * t) * (0.6 * 365)
    train_df_c.loc[i, 'FPull_4d06_365MPa_nosquare'] = round(f_pull_4d06_365MPa_nosquare, 1)
    f_pull_5d07_365MPa_nosquare = (np.pi/4) * np.square(4 * t) * (0.7 * 365)
    train_df_c.loc[i, 'FPull_4d07_365MPa_nosquare'] = round(f_pull_5d07_365MPa_nosquare, 1)

    f_pull_4d_310MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.8 * 440)
    train_df_c.loc[i, 'FPull_4d_440MPa'] = round(f_pull_4d_310MPa, 1)
    f_pull_5d_310MPa = (np.pi/4) * np.square(5 * np.sqrt(t)) * (0.8 * 440)
    train_df_c.loc[i, 'FPull_5d_440MPa'] = round(f_pull_5d_310MPa, 1)

# About (0.8·σ)

Different sources mention different values for the 0,8. If you go with Tresca or von Mises criterion you get 0,5-0,57 which gets approximated to 0,6.  With 45 degree shear plane theroy the relationship is 1/√2 which is 0,707. It is also typical for 0,5 to be used. austenitic stainless steels can range up to ~ 0.8 this material is low carbon steel though and doesn't fall into this category.

# Check Dataset

In [6]:
train_df_c.head()

,Sample ID,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm),Material,PullTest (N),...,FPull_4d_310MPa,FPull_5d_310MPa,FPull_4d_365MPa,FPull_5d_365MPa,FPull_4d_440MPa,FPull_5d_440MPa,FPull_4d_sig_06_365MPa,FPull_4d_sig_07_365MPa,FPull_4d06_365MPa_nosquare,FPull_4d07_365MPa_nosquare
0,1,35,200,0,6.82,1081.47,0.922,0.920,AISI 1010 carbon steel,2127.7,...,2867.1,4479.9,2953.9,3375.8,4069.5,6358.6,2531.9,2953.9,2329.3,2717.5
1,2,35,1500,0,52.25,2014.73,0.920,0.925,AISI 1010 carbon steel,5346.4,...,2867.1,4479.9,2953.9,3375.8,4069.5,6358.6,2531.9,2953.9,2329.3,2717.5
2,4,95,200,0,16.57,1321.93,0.912,0.924,AISI 1010 carbon steel,2350.4,...,2842.2,4441.0,2928.2,3346.5,4034.1,6303.3,2509.9,2928.2,2289.0,2670.5
3,5,95,200,0,41.42,1615.83,0.948,0.939,AISI 1010 carbon steel,2174.8,...,2926.4,4572.4,3014.9,3445.5,4153.5,6489.9,2584.2,3014.9,2426.5,2830.9
4,7,35,1500,0,63.82,1137.29,0.930,0.937,AISI 1010 carbon steel,3897.5,...,2898.3,4528.6,2986.0,3412.5,4113.7,6427.7,2559.4,2986.0,2380.2,2776.9


# Graph

In [19]:
x = train_df_c.index

fig = go.Figure()

groups = {
    "310MPa": {
        "cols": ["FPull_4d_310MPa", "FPull_5d_310MPa"],
        "colors": ["lightblue", "blue"]
    },
    "365MPa": {
        "cols": ["FPull_4d_365MPa", "FPull_5d_365MPa"],
        "colors": ["lightgreen", "green"]
    },
    "440MPa": {
        "cols": ["FPull_4d_440MPa", "FPull_5d_440MPa"],
        "colors": ["lightcoral", "red"]
    }
}

# --- Pull force groups ---
for info in groups.values():
    for col, col_color in zip(info["cols"], info["colors"]):
        fig.add_trace(
            go.Scatter(
                x=x,
                y=train_df_c[col],
                mode="markers",          # <-- no lines
                name=col,
                marker=dict(color=col_color, size=6)
            )
        )

# --- PullTest ---
fig.add_trace(
    go.Scatter(
        x=x,
        y=train_df_c["PullTest (N)"],
        mode="markers",
        name="PullTest (N)",
        marker=dict(color="grey", size=6)
    )
)

fig.update_layout(
    title="Pull Force Over Different MPa Values with PullTest (N) Comparison",
    xaxis_title="Row Number",
    yaxis_title="Values",
    template="seaborn",
    legend=dict(
        x=1.1,
        y=1,
        xanchor="left",
        borderwidth=1
    ),
    margin=dict(r=200)
)

fig.show()


In [24]:
fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=train_df_c["NuggetDiameter (mm)"],
        y=train_df_c["PullTest (N)"],
        mode="markers",
        name="PullForce vs NuggetDiameter",
        marker=dict(
            size=8,
            color="blue",
            opacity=0.7
        )
    )
)

fig2.update_layout(
    title="Correlation Between Pull Force and Nugget Diameter",
    xaxis_title="Nugget Diameter (mm)",
    yaxis_title="Pull Force (N)",
    template="seaborn"
)

fig2.show()

corr = train_df_c["NuggetDiameter (mm)"].corr(train_df_c["PullTest (N)"])
print("Pearson correlation:", corr)



Pearson correlation: 0.5613856609707731


In [23]:
x = train_df_c.index

fig = go.Figure()

groups = {
    "sig": {
        "cols": ["FPull_4d_sig_06_365MPa", "FPull_4d_sig_07_365MPa"],
        "colors": ["lightblue", "blue"]
    },
    "365MPa": {
        "cols": ["FPull_4d_365MPa", "FPull_5d_365MPa"],
        "colors": ["lightgreen", "green"]
    },
    "nosquare": {
        "cols": ["FPull_4d06_365MPa_nosquare", "FPull_4d07_365MPa_nosquare"],
        "colors": ["lightcoral", "red"]
    }
}

# --- Pull force groups ---
for info in groups.values():
    for col, col_color in zip(info["cols"], info["colors"]):
        fig.add_trace(
            go.Scatter(
                x=x,
                y=train_df_c[col],
                mode="markers",          # <-- no lines
                name=col,
                marker=dict(color=col_color, size=6)
            )
        )

# --- PullTest ---
fig.add_trace(
    go.Scatter(
        x=x,
        y=train_df_c["PullTest (N)"],
        mode="markers",
        name="PullTest (N)",
        marker=dict(color="grey", size=6)
    )
)

fig.update_layout(
    title="Pullforce estimation with different constants bevore σ ",
    xaxis_title="Row Number",
    yaxis_title="Values",
    template="seaborn",
    legend=dict(
        x=1.1,
        y=1,
        xanchor="left",
        borderwidth=1
    ),
    margin=dict(r=200)
)

fig.show()
